In [20]:
# -----------------------------
# 1. Importations
# -----------------------------
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from gensim.models import Word2Vec

In [21]:
# -----------------------------
# 2. Jeu de données
# -----------------------------
sentences = [
    ["je", "suis", "content"],
    ["je", "suis", "très", "heureux"],
    ["ce", "film", "est", "génial"],
    ["j", "adore", "ce", "livre"],
    ["ce", "cours", "est", "facile"],
    ["le", "professeur", "explique", "bien"],
    ["cette", "journée", "est", "parfaite"],
    ["je", "suis", "triste"],
    ["je", "suis", "fatigué"],
    ["ce", "film", "est", "mauvais"],
    ["je", "déteste", "les", "examens"],
    ["ce", "cours", "est", "difficile"],
    ["le", "travail", "est", "stressant"],
    ["cette", "journée", "est", "horrible"]
]

# Labels (1=positif, 0=négatif)
labels = [1,1,1,1,1,1,1, 0,0,0,0,0,0,0]

# -----------------------------
# 3. Entraîner Word2Vec
# -----------------------------
w2v_model = Word2Vec(
    sentences,
    vector_size=100,   # dimension des embeddings
    window=5,
    min_count=1,
    workers=4
)

# -----------------------------
# 4. Transformer phrases en vecteurs
# -----------------------------
max_len = 6  # longueur max d'une phrase

def sentence_to_vectors(sentence, model, max_len=6):
    vectors = []
    for word in sentence:
        if word in model.wv:
            vectors.append(model.wv[word])
        else:
            vectors.append(np.zeros(model.vector_size))
    # padding
    while len(vectors) < max_len:
        vectors.append(np.zeros(model.vector_size))
    return np.array(vectors[:max_len])

X = np.array([sentence_to_vectors(s, w2v_model, max_len=max_len) for s in sentences])
y = np.array(labels)

# Transformer en tenseurs PyTorch
X_torch = torch.tensor(X, dtype=torch.float32)
y_torch = torch.tensor(y, dtype=torch.long)

# -----------------------------
# 5. Modèle LSTM
# -----------------------------
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        last_hidden = h_n[-1]  # dernier état caché
        out = self.fc(last_hidden)
        return out

# -----------------------------
# 6. Modèle GRU
# -----------------------------
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        out, h_n = self.gru(x)
        last_hidden = h_n[-1]
        out = self.fc(last_hidden)
        return out

# -----------------------------
# 7. Paramètres
# -----------------------------
input_size = 100   # dimension Word2Vec
hidden_size = 64
num_classes = 2
num_epochs = 100
lr = 0.01

# Choisir LSTM ou GRU
model = LSTMModel(input_size, hidden_size, num_classes)
# model = GRUModel(input_size, hidden_size, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# -----------------------------
# 8. Entraînement
# -----------------------------
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_torch)
    loss = criterion(outputs, y_torch)
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        _, predicted = torch.max(outputs, 1)
        acc = (predicted == y_torch).sum().item() / len(y_torch)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Accuracy: {acc*100:.2f}%")

# -----------------------------
# 9. Test simple
# -----------------------------
model.eval()
test_sentence = ["je", "suis", "très", "heureux"]
test_vec = sentence_to_vectors(test_sentence, w2v_model, max_len)
test_tensor = torch.tensor(test_vec, dtype=torch.float32).unsqueeze(0)  # batch_size=1
pred = model(test_tensor)
pred_label = torch.argmax(pred, dim=1).item()
print(f"Phrase: {' '.join(test_sentence)}, Prédiction: {pred_label}")


Epoch [20/100], Loss: 0.4712, Accuracy: 92.86%
Epoch [40/100], Loss: 0.3555, Accuracy: 78.57%
Epoch [60/100], Loss: 0.0493, Accuracy: 100.00%
Epoch [80/100], Loss: 0.4462, Accuracy: 85.71%
Epoch [100/100], Loss: 0.6322, Accuracy: 50.00%
Phrase: je suis très heureux, Prédiction: 1
